# מעבדה 01 — שיווי משקל תרמי

במעבדה הזו שני גופים מחליפים ביניהם קוונטים של אנרגיה באקראי, אחד בכל פעם, עד שהם
מתייצבים על טמפרטורה משותפת — ואתם תמדדו כמה מהר, ובאיזו מידה של אמינות, זה קורה.

עבדו לפי הסדר. במקום שבו המחברת מבקשת מכם לנבא, כתבו את הניבוי בתא המיועד לכך **לפני**
הרצת התא הבא. זה אינו טקס: ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | שני גופים A ו-B, ממודלים כמוצקי איינשטיין בעלי $N_A$ ו-$N_B$ מתנדים |
| **דינמיקה** | בכל צעד מדלג קוונט אחד שנבחר באקראי, בהטיה לעבר עזיבת הגוף המלא יותר |
| **גבול** | סגור ומבודד כזוג; מספר הקוונטים הכולל (האנרגיה הכוללת) נשמר בדיוק |
| **צבר** | מיקרו-קנוני עבור המערכת המשותפת, בדיוק כמו במודול 8 |
| **מוזנח** | קצב הניסיונות הפיזיקלי (הזמן נמדד בצעדים ולא בשניות), מבנה מרחבי בתוך גוף |
| **תקף כאשר** | שני הגופים נמצאים במשטר הקלאסי של חלוקת האנרגיה השווה בטמפרטורה גבוהה |
| **אופני כישלון** | טמפרטורה נמוכה, גדלי קוונט שונים בין הגופים, מעט מדי מתנדים |

כל הפיזיקה נמצאת ב-`thermolab.equilibrium` — פתחו וקראו אותה. שום דבר בקורס הזה אינו מוסתר
בתוך תשתית תוכנה.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import equilibrium
from thermolab.constants import K_B
from thermolab.validation import scaling_exponent, seed_study

QUANTUM = 20.0 * K_B  # an arbitrary but fixed energy quantum, shared by both bodies

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"k_B = {K_B:.6e} J/K")
print(f"quantum = {QUANTUM:.6e} J")

## חלק 1 — לצפות בשני גופים מוצאים טמפרטורה משותפת

התחילו בשני גופים בגודל בינוני, בטמפרטורות שונות מאוד זו מזו.

In [ ]:
state = equilibrium.from_temperatures(150, 50, 500.0, 250.0, QUANTUM)
n_steps = 20000
result = equilibrium.simulate_energy_exchange(state, n_steps, rng)

t_eq = equilibrium.equilibrium_temperature(
    state.heat_capacity_a, state.temperature_a, state.heat_capacity_b, state.temperature_b
)
predicted_gap = equilibrium.predicted_relaxation(state, result.steps)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(result.steps, result.temperature_a, lw=0.8, color="crimson", label="body A")
ax.plot(result.steps, result.temperature_b, lw=0.8, color="steelblue", label="body B")
ax.plot(result.steps, t_eq + predicted_gap, lw=1.4, ls="--", color="black",
        label="predicted (analytic)")
ax.axhline(t_eq, color="grey", ls=":", lw=1.0, label="T_eq")
ax.set_xlabel("step")
ax.set_ylabel("temperature (K)")
ax.set_title(f"N_A = {state.n_a}, N_B = {state.n_b}: two bodies relaxing to T_eq")
ax.legend()
plt.tight_layout()
plt.show()

print(f"T_eq predicted            {t_eq:.3f} K")
print(f"T_A after {n_steps} steps   {result.temperature_a[-1]:.3f} K")
print(f"T_B after {n_steps} steps   {result.temperature_b[-1]:.3f} K")
print(f"relaxation time tau = {equilibrium.relaxation_time(state):.1f} steps")

הפער בין שתי העקומות מצטמצם בהדרגה, בלי להחליף סימן אף פעם.

### לנבא

לפני הרצת התא הבא, רשמו מה אתם מצפים שיקרה אם גוף B ייעשה *קטן* פי עשרה מאשר בריצה הזו
(כך ששני הגופים יהיו שונים מאוד בגודלם) בעוד טמפרטורות ההתחלה נשארות זהות. האם טמפרטורת
שיווי המשקל נעה לעבר $T_A(0)$, לעבר $T_B(0)$, או נשארת באמצע? היו ספציפיים.

**הניבוי שלכם:**

*(כתבו כאן לפני הרצת התא הבא)*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)

for ax, (n_a, n_b) in zip(axes, ((30, 10), (300, 100), (3000, 1000)), strict=True):
    s = equilibrium.from_temperatures(n_a, n_b, 500.0, 250.0, QUANTUM)
    tau = equilibrium.relaxation_time(s)
    steps_needed = int(6 * tau)
    run = equilibrium.simulate_energy_exchange(s, steps_needed, rng)
    gap = run.temperature_a - run.temperature_b
    delta_t0 = s.temperature_a - s.temperature_b

    ax.plot(run.steps / tau, gap / delta_t0, lw=1.0, color="steelblue")
    ax.plot(run.steps / tau, equilibrium.predicted_relaxation(s, run.steps) / delta_t0,
            lw=1.2, ls="--", color="crimson")
    ax.set_xlabel("step / tau")
    ax.set_title(f"N = {n_a + n_b}")

axes[0].set_ylabel(r"$(T_A - T_B) \,/\, (T_{A,0} - T_{B,0})$")
plt.tight_layout()
plt.show()

כל פאנל מתחיל בפער יחסי של בדיוק $1$ ודועך לאורך אותה עקומה מקווקוות, יהיה גודל המערכת
אשר יהיה. מה שמשתנה הוא רק הפיזור סביב העקומה ההיא — מערכות קטנות יותר נודדות רחוק יותר מן
הניבוי החלק, מאותה סיבה שבגללה הלחץ של מדגם גז קטן יותר רוטט יותר.

## חלק 2 — האם טמפרטורת שיווי המשקל היא באמת הממוצע המשוקלל?

מדדו, אל תניחו. שנו את היחס בין גודלי שני הגופים והשוו את הטמפרטורה בזמן ארוך לנוסחה
הממוסגרת $T_{eq} = (C_A T_{A,0} + C_B T_{B,0})/(C_A + C_B)$.

In [ ]:
def measure_equilibrium_temperature(n_a, n_b, t_a0, t_b0, rng, n_tau=8):
    s = equilibrium.from_temperatures(n_a, n_b, t_a0, t_b0, QUANTUM)
    n_steps = int(n_tau * equilibrium.relaxation_time(s))
    run = equilibrium.simulate_energy_exchange(s, n_steps, rng)
    return float(run.temperature_a[-1]), s


configs = [(150, 150, 500.0, 250.0), (150, 50, 500.0, 250.0), (50, 150, 500.0, 250.0)]
for n_a, n_b, t_a0, t_b0 in configs:
    measured, s = measure_equilibrium_temperature(n_a, n_b, t_a0, t_b0, rng)
    predicted = equilibrium.equilibrium_temperature(
        s.heat_capacity_a, s.temperature_a, s.heat_capacity_b, s.temperature_b
    )
    print(f"N_A={n_a:4d} N_B={n_b:4d}   measured {measured:7.2f} K   predicted {predicted:7.2f} K")

כאשר שני הגופים באותו גודל, שיווי המשקל שוכן באמצע, ב-$375\ K$. הפכו את גוף A לגדול פי
שלושה מגוף B (או להפך), וטמפרטורת שיווי המשקל נעה לעבר הגוף בעל קיבול החום הגדול יותר —
בדיוק כפי שנוסחת הממוצע המשוקלל חוזה, ולא כלל כמו ממוצע פשוט של שתי טמפרטורות ההתחלה.

## חלק 3 — עד כמה צפויה ההתקרבות אל שיווי המשקל?

*עקומת* הרלקסציה זהה בכל גודל (חלק 1). מה שמשתנה עם הגודל הוא עד כמה הדוקות מתקבצות ריצות
בודדות סביבה — אותה התייצבות התלויה ב-$N$ שפוגשים לאורך כל הקורס.

In [ ]:
def relative_spread_at_one_tau(n_a, n_b, n_samples, rng):
    s = equilibrium.from_temperatures(n_a, n_b, 500.0, 250.0, QUANTUM)
    tau = equilibrium.relaxation_time(s)
    checkpoint = max(int(round(tau)), 1)
    gaps = []
    for _ in range(n_samples):
        run = equilibrium.simulate_energy_exchange(s, checkpoint, rng)
        gaps.append(float(run.temperature_a[-1] - run.temperature_b[-1]))
    gaps = np.array(gaps)
    return float(gaps.std(ddof=1) / abs(gaps.mean())), s.total_quanta


sizes = [(15, 5), (50, 17), (150, 50), (500, 167), (1500, 500)]
spreads, totals = [], []
for n_a, n_b in sizes:
    spread, q_total = relative_spread_at_one_tau(n_a, n_b, 30, rng)
    spreads.append(spread)
    totals.append(q_total)
    print(f"N_A={n_a:5d} N_B={n_b:5d}   Q={q_total:6d}   relative spread = {spread:.4f}")

exponent = scaling_exponent(totals, spreads)
print(f"\nfitted exponent = {exponent:.3f}   (theory: roughly -0.5)")

plt.figure(figsize=(6, 4.5))
plt.loglog(totals, spreads, "o", label="measured")
plt.loglog(totals, spreads[0] * (np.array(totals) / totals[0]) ** -0.5, "-", label=r"$Q^{-1/2}$")
plt.xlabel("Q (total quanta)")
plt.ylabel("relative spread of the gap, one tau in")
plt.title(f"fitted slope {exponent:.2f}")
plt.legend()
plt.tight_layout()
plt.show()

## חלק 4 — בדיקות אוטומטיות

סימולציה שלא בדקתם היא תמונה, לא ראיה. אלה בדיוק אותן טענות הרצות בחבילת הבדיקות של
הפרויקט.

In [ ]:
check_state = equilibrium.from_temperatures(150, 50, 500.0, 250.0, QUANTUM)
check_steps = int(8 * equilibrium.relaxation_time(check_state))
check_run = equilibrium.simulate_energy_exchange(check_state, check_steps, np.random.default_rng(1))

# 1. Energy conservation -- every step only relabels which body owns one quantum.
assert np.all(check_run.q_a + check_run.q_b == check_state.total_quanta)

# 2. The equilibrium temperature, across independent seeds and with an honest error bar.
target = equilibrium.equilibrium_temperature(
    check_state.heat_capacity_a, check_state.temperature_a,
    check_state.heat_capacity_b, check_state.temperature_b,
)


def final_temperature_a(r):
    return equilibrium.simulate_energy_exchange(check_state, check_steps, r).temperature_a[-1]


study = seed_study(final_temperature_a, n_seeds=12)
assert study.agrees_with(target, n_sigma=3.5)

# 3. The relaxation curve, one relaxation time in.
checkpoint = int(equilibrium.relaxation_time(check_state))
predicted = float(equilibrium.predicted_relaxation(check_state, np.array([checkpoint]))[0])


def gap_after_one_tau(r):
    run = equilibrium.simulate_energy_exchange(check_state, checkpoint, r)
    return float(run.temperature_a[-1] - run.temperature_b[-1])


gap_study = seed_study(gap_after_one_tau, n_seeds=16)
assert gap_study.agrees_with(predicted, n_sigma=3.5)

print(f"T_eq         measured {study.mean:.3f} +/- {study.standard_error:.3f}   vs   {target:.3f}")
print(
    f"gap at tau   measured {gap_study.mean:.3f} +/- {gap_study.standard_error:.3f}   "
    f"vs   {predicted:.3f}"
)
print("\nall checks passed")

## חלק 5 — לחקור בעצמכם

המחוונים שלהלן מאפשרים לכם לשנות בחופשיות את גודלם ואת טמפרטורות ההתחלה של שני הגופים.
שני ניסויים שכדאי לבצע:

1. הפכו את שני הגופים לשונים מאוד בגודלם וצפו כמה מעט זזה הטמפרטורה של הגדול שבהם — זו
   הסיבה שמדחום (קטן) כמעט אינו מפריע לדבר שהוא מודד (גדול).
2. מצאו גודל קטן דיו כדי שעקומת הרלקסציה תיראה רועשת באופן נראה לעין ולא חלקה, ואמרו באיזה
   קנה מידה השתמשתם כדי להכריע מהו "רועש".

In [ ]:
import ipywidgets as widgets


def explore(n_a=150, n_b=50, t_a0=500.0, t_b0=250.0, n_tau=6.0):
    s = equilibrium.from_temperatures(n_a, n_b, t_a0, t_b0, QUANTUM)
    tau = equilibrium.relaxation_time(s)
    n_steps = max(int(n_tau * tau), 10)
    run = equilibrium.simulate_energy_exchange(s, n_steps, np.random.default_rng(0))
    t_eq = equilibrium.equilibrium_temperature(
        s.heat_capacity_a, s.temperature_a, s.heat_capacity_b, s.temperature_b
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(run.steps, run.temperature_a, lw=1.0, color="crimson", label="body A")
    ax.plot(run.steps, run.temperature_b, lw=1.0, color="steelblue", label="body B")
    ax.axhline(t_eq, color="grey", ls=":", lw=1.0, label="T_eq")
    ax.set_xlabel("step")
    ax.set_ylabel("temperature (K)")
    ax.set_title(f"tau = {tau:.0f} steps, T_eq = {t_eq:.1f} K")
    ax.legend()
    plt.tight_layout()
    plt.show()


widgets.interact_manual(
    explore,
    n_a=widgets.IntSlider(min=5, max=3000, step=5, value=150, description="N_A"),
    n_b=widgets.IntSlider(min=5, max=3000, step=5, value=50, description="N_B"),
    t_a0=widgets.FloatSlider(min=100, max=900, step=10, value=500.0, description="T_A0 (K)"),
    t_b0=widgets.FloatSlider(min=100, max=900, step=10, value=250.0, description="T_B0 (K)"),
    n_tau=widgets.FloatSlider(min=1, max=10, step=0.5, value=6.0, description="steps (x tau)"),
);

## חלק 6 — ניסוי אמיתי: עקומת הקירור שלכם

כל מה שמעל הנקודה הזו הוא סימולציה המסכימה עם גזירה, וזה בוחן את המתמטיקה ולא את העולם.
החלק הזה הוא הבדיקה מן הסוג האחר, והוא זקוק למטבח ולא למחשב.

**המדידה.** מלאו ספל במים חמים — מן הברז, או מקומקום שהונח לעמוד דקה; לא רותחים. העמידו בו
מדחום מטבח או מדחום מזון, קראו את טמפרטורת החדר פעם אחת ורשמו אותה, ואז קראו את טמפרטורת
המים כל שתי דקות במשך עשרים עד שלושים דקות. תריסר קריאות מספיקות בהחלט. השאירו את הספל
במנוחה, הרחק מזרמי אוויר ומאור שמש: למודל יש מגע תרמי אחד ואין בו רוח.

**מה המודל חוזה.** לא שהטמפרטורה יורדת בקו ישר, ולא שהיא יורדת לאפס — ה*פער*
$T - T_{\text{room}}$ דועך מעריכית עם קבוע זמן יחיד, וזהו חוק הקירור של ניוטון כשהחדר
משחק את תפקידו של גוף גדול כל כך עד שטמפרטורתו שלו אינה זזה לעולם. בשרטוט של
$\ln(T - T_{\text{room}})$ מול הזמן, הניבוי הזה הוא קו ישר בשיפוע $-1/\tau$, והתאמת הקו
הזה היא כל המדידה.

עקמומיות בו היא התוצאה המעניינת, לא הכושלת. אידוי נושא אנרגיה החוצה כחום כמוס, ערוץ שלמודל
הזה אין כלל, וספל אמיתי מאבד חום בהולכה, בהסעה ובקרינה בעת ובעונה אחת — ולכן המוליכות
שמאחורי ה-$\tau$ שהתאמתם שייכת לספל ההוא בחדר ההוא, ולא למים.

In [ ]:
# Example values only: these three are invented — a plausible-looking mug of water, so that the
# cell runs before you have data of your own. They are nobody's measurement. Replace all three
# with your own readings; nothing below them needs changing.
time_min = np.array([0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0, 18.0, 20.0, 24.0, 28.0])
temp_c = np.array([78.1, 73.2, 69.0, 65.1, 61.7, 58.4, 55.5, 52.8, 50.3, 48.2, 46.1, 42.4, 39.3])
t_room_c = 22.0

gap = temp_c - t_room_c
assert np.all(gap > 0), "every reading must sit above room temperature -- check t_room_c"

# log(T - T_room) = log(gap at t=0) - t/tau, so one straight-line fit through the logged gap
# delivers the time constant. No curve fitter is needed, and none is available in the browser.
slope, intercept = np.polyfit(time_min, np.log(gap), 1)
tau_minutes = -1.0 / slope
gap_0 = float(np.exp(intercept))

fit_time = np.linspace(time_min.min(), time_min.max(), 200)
fit_temp = t_room_c + gap_0 * np.exp(-fit_time / tau_minutes)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(time_min, temp_c, "o", color="crimson", label="your readings")
ax.plot(fit_time, fit_temp, lw=1.4, ls="--", color="black", label="fitted exponential")
ax.axhline(t_room_c, color="grey", ls=":", lw=1.0, label="T_room")
ax.set_xlabel("time (min)")
ax.set_ylabel("temperature (°C)")
ax.set_title(f"cooling curve: tau = {tau_minutes:.1f} min")
ax.legend()
plt.tight_layout()
plt.show()

print(f"relaxation time tau = {tau_minutes:.1f} min")
print(f"gap at t = 0        fitted {gap_0:.1f} K   measured {gap[0]:.1f} K")
print(f"half the gap closes in {tau_minutes * np.log(2):.1f} min")

## בדקו את הבנתכם

הריצו את התא שלהלן לחידון עם בדיקה אוטומטית. אותן שאלות, בתוספת הסברים כתובים לכל אפשרות,
נמצאות בעמוד המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "01-equilibrium.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת מהשאלות, בתא שלהלן.

1. מה ניבאתם בחלק 1 שהתברר כשגוי, ומה בדיוק היה הפגם בהיגיון שלכם?
2. המודל הזה מחליף קוונטים זהים בין שני מוצקי איינשטיין אידיאליים. נקבו במסקנה אחת מהיום
   שלכן **אינה** מבוססת ביחס למגע תרמי אמיתי, אף שהמספרים התאימו לגזירה.
3. הסבירו, בלי משוואות, מדוע קיבול חום גדול יותר מושך את טמפרטורת שיווי המשקל לעבר ערך
   ההתחלה שלו עצמו.

**התשובות שלכם:**

1.
2.
3.